[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/11.%20The%20Dynamic%20Berth%20Allocation%20Problem/P11-Tier-4_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 11. The Dynamic Berth Allocation Problem
## Tier 4 — Reinforcement Learning Augmentation

### Goal and scope
This notebook models dynamic berth allocation as a finite-horizon Markov decision process (MDP) and trains an agent to choose vessel–berth assignments over a sequence of decisions. The inputs are a fixed list of vessels, their arrival times, lengths, deadlines, berth-specific processing times, and berth capacities. The outputs are a learned policy, episode-level training metrics, and a record of decisions made during evaluation.

The example uses deterministic vessel data and processing times. Arrival uncertainty, weather, spatial berth positions, stochastic processing, and interactions with other port operations are outside this environment. The fixed example therefore illustrates the learning loop; it does not establish performance on unseen or stochastic instances.

### MDP intuition and formal model
At each decision point, the agent observes the current time, berth availability, and each vessel's status. It can assign an arrived vessel to a feasible, currently available berth, or wait for one time unit. Waiting is necessary when a vessel has not arrived or no feasible berth is free. An assignment completes after its berth-specific processing time.

An MDP is described by states \(s\), actions \(a\), transition probabilities \(P(s'\mid s,a)\), rewards \(r(s,a,s')\), and a discount factor \(\gamma\). Here, the example is deterministic once an action is chosen, so its transition probability is concentrated on the resulting next state. The state contains enough information to determine feasible actions and future transitions for this fixed instance.

For an assignment of vessel \(i\) to berth \(j\), let \(t\) be the current time, \(a_i\) the vessel's arrival time, \(d_i\) its desired departure time, \(p_{ij}\) its processing time at berth \(j\), and \(C_{ij}=\max(t,a_i)+p_{ij}\) its completion time. The immediate assignment reward is

$$
r(s,a,s')=-w_W(C_{ij}-p_{ij}-a_i)-w_Pp_{ij}-w_D\max(0,C_{ij}-d_i)+w_U p_{ij}.
$$

The first term penalizes waiting, the second processing duration, the third tardiness, and the last gives a utilization-related bonus for processing work. A wait action receives reward \(-0.1\). At episode termination, the reward also includes \(-0.1\) times the sum of completed vessels' service durations (completion time minus arrival time). These are reward components, not a direct guarantee that the learned policy minimizes a single unweighted scheduling objective.

The action-value function and tabular Q-learning update are

$$
Q^*(s,a)=\mathbb{E}\left[r(s,a,s')+\gamma\max_{a'\in A(s')}Q^*(s',a')\right],
$$

$$
Q(s,a)\leftarrow Q(s,a)+\alpha\left[r+\gamma\max_{a'\in A(s')}Q(s',a')-Q(s,a)\right].
$$

Here, \(A(s')\) is the set of valid actions in the next state, \(\alpha\) is the learning rate, and \(\gamma\) discounts future rewards. At a terminal state, the future-value term is zero. The implementation uses an epsilon-greedy policy: with probability \(\epsilon\), it explores by selecting a valid action; otherwise, it chooses the valid action with the greatest estimated value.

### Method and interpretation
The notebook retains the Q-learning solution approach. When PyTorch is available, the same action-value learning idea is represented by a Deep Q-Network (DQN), with experience replay and a target network; otherwise, discretized states are stored in a tabular Q-learning agent. The tabular representation is approximate because continuous state components are rounded before they form a table key. DQN generalizes through a neural network but does not guarantee an optimal policy. Neither implementation guarantees convergence for this finite training run.

Training repeats episodes, records rewards and service counts, and gradually reduces exploration. Evaluation sets exploration to zero and tests the learned policy on the same fixed instance. The included EDF and random policies are diagnostic reference policies, not part of the training method. A reward trend or policy comparison should be interpreted as empirical evidence for these settings, not as a proof of optimality or generalization.

In [ ]:
import copy
import random
import time
from collections import defaultdict, deque
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    DEEP_LEARNING_AVAILABLE = True
    print("PyTorch is available; the DQN agent will be used.")
except ImportError:
    DEEP_LEARNING_AVAILABLE = False
    print("PyTorch is unavailable; the tabular Q-learning agent will be used.")

print("Libraries imported successfully.")

## 1. Define the instance

Vessels have arrival times, desired departure times, lengths, and processing times that depend on the chosen berth. A berth can serve a vessel only if its length capacity is sufficient. Times and processing durations are measured in hours; lengths and capacities are measured in metres. The TEU field is descriptive and is not used by the current reward or feasibility rules.

In [ ]:
@dataclass
class Vessel:
    """A vessel and its berth-dependent service requirements."""
    id: int
    arrival_time: int
    desired_departure: int
    length: int
    teu_capacity: int
    processing_times: Dict[int, int]


@dataclass
class Berth:
    """A berth with a maximum vessel length in metres."""
    id: int
    capacity: int


vessels = [
    Vessel(1, 0, 10, 200, 5000, {1: 4, 2: 5}),
    Vessel(2, 2, 12, 250, 8000, {1: 3, 2: 4}),
    Vessel(3, 4, 15, 300, 12000, {1: 6, 2: 5}),
]

berths = [Berth(1, 350), Berth(2, 400)]

print(f"Problem: {len(vessels)} vessels and {len(berths)} berths")
for vessel in vessels:
    print(
        f"  Vessel {vessel.id}: length={vessel.length} m, "
        f"TEU={vessel.teu_capacity}, arrival={vessel.arrival_time} h, "
        f"deadline={vessel.desired_departure} h"
    )
for berth in berths:
    print(f"  Berth {berth.id}: length capacity={berth.capacity} m")

## 2. Build the MDP environment

The environment stores the state needed to make decisions: current time, each berth's next available time, and whether each vessel has not arrived, is waiting, or has been served. Vessel-specific waiting time and urgency are included as state features. A fixed action index represents a particular vessel–berth pair; this keeps an action's meaning stable even when the waiting queue changes.

The action space contains one action for each vessel–berth pair plus a wait action. Only feasible assignments and waiting are returned as valid actions. After waiting, time advances by one hour and newly arrived vessels enter the queue. An episode ends when every vessel is served or the time horizon is reached.

In [ ]:
class DynamicBerthEnvironment:
    """Finite-horizon berth-allocation environment for Q-learning."""

    def __init__(
        self,
        vessels: List[Vessel],
        berths: List[Berth],
        max_time_horizon: int = 24,
    ):
        self.vessels = vessels
        self.berths = berths
        self.max_time = max_time_horizon
        self.berth_dict = {berth.id: berth for berth in berths}
        self.n_vessels = len(vessels)
        self.n_berths = len(berths)
        self.action_space_n = self.n_vessels * self.n_berths + 1
        self.wait_action = self.action_space_n - 1
        self.state_size = 1 + 2 * self.n_berths + 6 * self.n_vessels
        self.waiting_weight = 1.0
        self.processing_weight = 1.0
        self.delay_weight = 2.0
        self.utilization_weight = 0.1
        self.reset()

    def reset(self) -> np.ndarray:
        """Restore the initial condition and return its state."""
        self.current_time = 0
        self.waiting_queue = []
        self.berth_availability = {berth.id: 0 for berth in self.berths}
        self.completed_vessels = []
        self.assignment_history = []
        self.update_waiting_queue()
        return self.get_state()

    def get_state(self) -> np.ndarray:
        """Encode time, berth availability, and vessel status as a vector."""
        state = [self.current_time / self.max_time]
        for berth in self.berths:
            remaining = max(
                0, self.berth_availability[berth.id] - self.current_time
            )
            state.extend([remaining / self.max_time, float(remaining == 0)])

        for vessel in self.vessels:
            not_arrived = float(
                vessel not in self.waiting_queue
                and vessel not in self.completed_vessels
                and vessel.arrival_time > self.current_time
            )
            waiting = float(vessel in self.waiting_queue)
            completed = float(vessel in self.completed_vessels)
            waiting_time = (
                max(0, self.current_time - vessel.arrival_time) / self.max_time
                if waiting else 0.0
            )
            urgency = (
                max(0, vessel.desired_departure - self.current_time) / self.max_time
                if waiting else 0.0
            )
            size = vessel.teu_capacity / 15000
            state.extend(
                [not_arrived, waiting, completed, waiting_time, urgency, size]
            )
        return np.asarray(state, dtype=np.float32)

    def update_waiting_queue(self) -> None:
        """Make arrived, unserved vessels available for assignment."""
        for vessel in self.vessels:
            if (
                vessel.arrival_time <= self.current_time
                and vessel not in self.waiting_queue
                and vessel not in self.completed_vessels
            ):
                self.waiting_queue.append(vessel)

    def get_valid_actions(self) -> List[int]:
        """Return feasible vessel–berth assignments and the wait action."""
        valid_actions = [self.wait_action]
        for vessel_idx, vessel in enumerate(self.vessels):
            if vessel not in self.waiting_queue:
                continue
            for berth_idx, berth in enumerate(self.berths):
                if (
                    vessel.length <= berth.capacity
                    and self.berth_availability[berth.id] <= self.current_time
                ):
                    valid_actions.append(vessel_idx * self.n_berths + berth_idx)
        return valid_actions

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, Dict]:
        """Apply one valid action and return state, reward, terminal flag, info."""
        if action not in self.get_valid_actions():
            raise ValueError(f"Action {action} is not valid in the current state.")

        reward = 0.0
        info = {}
        if action == self.wait_action:
            self.current_time += 1
            self.update_waiting_queue()
            reward = -0.1
            info["action_type"] = "wait"
        else:
            vessel_idx, berth_idx = divmod(action, self.n_berths)
            vessel = self.vessels[vessel_idx]
            berth = self.berths[berth_idx]
            start_time = max(self.current_time, vessel.arrival_time)
            processing_time = vessel.processing_times[berth.id]
            completion_time = start_time + processing_time
            waiting_time = start_time - vessel.arrival_time
            delay = max(0, completion_time - vessel.desired_departure)

            reward = (
                -self.waiting_weight * waiting_time
                -self.processing_weight * processing_time
                -self.delay_weight * delay
                +self.utilization_weight * processing_time
            )
            self.berth_availability[berth.id] = completion_time
            self.completed_vessels.append(vessel)
            self.waiting_queue.remove(vessel)
            assignment = {
                "time": self.current_time,
                "vessel_id": vessel.id,
                "berth_id": berth.id,
                "start_time": start_time,
                "completion_time": completion_time,
                "waiting_time": waiting_time,
                "processing_time": processing_time,
            }
            self.assignment_history.append(assignment)
            info.update(assignment)
            info["action_type"] = "assign"

        done = (
            len(self.completed_vessels) == len(self.vessels)
            or self.current_time >= self.max_time
        )
        if done:
            service_time = sum(
                item["completion_time"]
                - self.vessels[ next(
                    i for i, vessel in enumerate(self.vessels)
                    if vessel.id == item["vessel_id"]
                )].arrival_time
                for item in self.assignment_history
            )
            reward -= 0.1 * service_time
            info.update(
                episode_complete=True,
                vessels_served=len(self.completed_vessels),
                total_time=self.current_time,
            )
        return self.get_state(), reward, done, info

    def render(self) -> None:
        """Print a compact view of the current environment state."""
        print(f"Time: {self.current_time} h")
        print(f"Waiting vessel IDs: {[v.id for v in self.waiting_queue]}")
        print(f"Berth availability: {self.berth_availability}")
        print(f"Completed vessel IDs: {[v.id for v in self.completed_vessels]}")

print("Environment class defined successfully.")

## 3. Define the Q-learning agents

The tabular agent rounds state features to one decimal place and uses the resulting string as a sparse dictionary key. This makes a continuous-valued state usable in a table, but different underlying states can map to the same key. Its update uses the maximum value over valid next actions, which prevents infeasible assignments from influencing the learned target.

When PyTorch is available, the DQN approximates action values with a neural network. Experience replay reuses stored transitions, and a target network supplies a more stable estimate of future value. The next-state valid-action set is stored with each transition so that the DQN target also excludes infeasible actions. Both agents use the same environment, reward, and epsilon-greedy action selection.

In [ ]:
if DEEP_LEARNING_AVAILABLE:
    class DQNNetwork(nn.Module):
        """Neural approximation of action values."""

        def __init__(self, state_size: int, action_size: int, hidden_size: int = 128):
            super().__init__()
            self.fc1 = nn.Linear(state_size, hidden_size)
            self.fc2 = nn.Linear(hidden_size, hidden_size)
            self.fc3 = nn.Linear(hidden_size, hidden_size // 2)
            self.fc4 = nn.Linear(hidden_size // 2, action_size)
            self.dropout = nn.Dropout(0.2)

        def forward(self, state):
            state = torch.relu(self.fc1(state))
            state = self.dropout(state)
            state = torch.relu(self.fc2(state))
            state = self.dropout(state)
            state = torch.relu(self.fc3(state))
            return self.fc4(state)


    class DQNAgent:
        """DQN with replay memory, valid-action masking, and a target network."""

        def __init__(self, state_size: int, action_size: int, learning_rate: float = 0.001):
            self.state_size = state_size
            self.action_size = action_size
            self.learning_rate = learning_rate
            self.gamma = 0.99
            self.epsilon = 1.0
            self.epsilon_min = 0.01
            self.epsilon_decay = 0.995
            self.batch_size = 32
            self.memory = deque(maxlen=10000)
            self.target_update_freq = 10
            self.training_step = 0
            self.loss_history = []
            self.q_network = DQNNetwork(state_size, action_size)
            self.target_network = DQNNetwork(state_size, action_size)
            self.target_network.load_state_dict(self.q_network.state_dict())
            self.optimizer = optim.Adam(self.q_network.parameters(), lr=learning_rate)

        def remember(self, state, action, reward, next_state, done, next_valid_actions):
            self.memory.append(
                (state, action, reward, next_state, done, next_valid_actions)
            )

        def act(self, state: np.ndarray, valid_actions: Optional[List[int]] = None) -> int:
            valid_actions = list(valid_actions or range(self.action_size))
            if random.random() <= self.epsilon:
                return random.choice(valid_actions)
            was_training = self.q_network.training
            self.q_network.eval()
            with torch.no_grad():
                values = self.q_network(torch.as_tensor(state, dtype=torch.float32))
            self.q_network.train(was_training)
            masked = torch.full_like(values, float("-inf"))
            masked[valid_actions] = values[valid_actions]
            return int(masked.argmax().item())

        def replay(self):
            if len(self.memory) < self.batch_size:
                return
            batch = random.sample(self.memory, self.batch_size)
            states = torch.as_tensor(np.asarray([item[0] for item in batch]), dtype=torch.float32)
            actions = torch.as_tensor([item[1] for item in batch], dtype=torch.long)
            rewards = torch.as_tensor([item[2] for item in batch], dtype=torch.float32)
            next_states = torch.as_tensor(np.asarray([item[3] for item in batch]), dtype=torch.float32)
            dones = torch.as_tensor([item[4] for item in batch], dtype=torch.bool)
            current_q = self.q_network(states).gather(1, actions[:, None]).squeeze(1)
            with torch.no_grad():
                next_values = self.target_network(next_states)
                targets = rewards.clone()
                for row, item in enumerate(batch):
                    if not item[4] and item[5]:
                        targets[row] += self.gamma * next_values[row, item[5]].max()
            loss = nn.MSELoss()(current_q, targets)
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), 1.0)
            self.optimizer.step()
            self.loss_history.append(float(loss.item()))
            self.training_step += 1
            if self.training_step % self.target_update_freq == 0:
                self.target_network.load_state_dict(self.q_network.state_dict())
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

        def save_model(self, filepath: str):
            torch.save(
                {
                    "q_network_state_dict": self.q_network.state_dict(),
                    "target_network_state_dict": self.target_network.state_dict(),
                    "optimizer_state_dict": self.optimizer.state_dict(),
                    "epsilon": self.epsilon,
                    "training_step": self.training_step,
                },
                filepath,
            )

        def load_model(self, filepath: str):
            checkpoint = torch.load(filepath, map_location="cpu")
            self.q_network.load_state_dict(checkpoint["q_network_state_dict"])
            self.target_network.load_state_dict(checkpoint["target_network_state_dict"])
            self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            self.epsilon = checkpoint["epsilon"]
            self.training_step = checkpoint["training_step"]

else:
    class TabularQAgent:
        """Sparse tabular Q-learning agent with rounded state features."""

        def __init__(self, state_size: int, action_size: int, learning_rate: float = 0.1):
            self.state_size = state_size
            self.action_size = action_size
            self.learning_rate = learning_rate
            self.gamma = 0.99
            self.epsilon = 1.0
            self.epsilon_min = 0.01
            self.epsilon_decay = 0.995
            self.q_table = defaultdict(lambda: np.zeros(action_size, dtype=float))

        def _state_to_key(self, state: np.ndarray) -> str:
            discretized = np.round(state * 10).astype(int)
            return ",".join(map(str, discretized))

        def act(self, state: np.ndarray, valid_actions: Optional[List[int]] = None) -> int:
            valid_actions = list(valid_actions or range(self.action_size))
            if random.random() <= self.epsilon:
                return random.choice(valid_actions)
            values = self.q_table[self._state_to_key(state)]
            return max(valid_actions, key=lambda action: values[action])

        def remember(self, state, action, reward, next_state, done, next_valid_actions):
            state_key = self._state_to_key(state)
            next_key = self._state_to_key(next_state)
            current_q = self.q_table[state_key][action]
            next_values = self.q_table[next_key]
            future_value = (
                max(next_values[a] for a in next_valid_actions)
                if not done and next_valid_actions else 0.0
            )
            target = reward + self.gamma * future_value
            self.q_table[state_key][action] += self.learning_rate * (target - current_q)
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

        def replay(self):
            """Tabular learning updates occur when each transition is observed."""
            return

print("Q-learning agent classes defined.")

## 4. Train and evaluate

Each transition is used immediately by tabular Q-learning and stored in replay memory by DQN. The DQN replay step is called during the episode, after a transition has entered memory, rather than only once after an episode. Evaluation temporarily sets \(\epsilon=0\), so choices are greedy among valid actions; the original exploration setting is restored afterward.

An episode may terminate because every vessel is served or because the time horizon is reached. In the latter case, the service count makes incomplete service visible. Reward, episode length, and vessels served are complementary measurements: a reward value alone does not identify which operational outcomes produced it.

In [ ]:
def train_rl_agent(env, episodes: int = 500, max_steps: int = 100):
    """Train a Q-learning agent and return the agent and episode metrics."""
    agent_class = DQNAgent if DEEP_LEARNING_AVAILABLE else TabularQAgent
    agent = agent_class(env.state_size, env.action_space_n)
    print("Reinforcement learning training")
    print(f"Agent: {'DQN' if DEEP_LEARNING_AVAILABLE else 'Tabular Q-learning'}")
    print(f"Episodes: {episodes}")

    history = {
        "episode_rewards": [],
        "episode_lengths": [],
        "vessels_served": [],
        "epsilon_history": [],
    }
    for episode in range(episodes):
        state = env.reset()
        total_reward = 0.0
        info = {}
        for step in range(max_steps):
            valid_actions = env.get_valid_actions()
            action = agent.act(state, valid_actions)
            next_state, reward, done, info = env.step(action)
            next_valid_actions = env.get_valid_actions() if not done else []
            agent.remember(
                state, action, reward, next_state, done, next_valid_actions
            )
            agent.replay()
            state = next_state
            total_reward += reward
            if done:
                break
        history["episode_rewards"].append(total_reward)
        history["episode_lengths"].append(step + 1)
        history["vessels_served"].append(info.get("vessels_served", 0))
        history["epsilon_history"].append(agent.epsilon)
        if episode % 50 == 0 or episode == episodes - 1:
            recent = history["episode_rewards"][-50:]
            print(
                f"Episode {episode:3d}: reward={total_reward:8.2f}, "
                f"recent mean={np.mean(recent):8.2f}, "
                f"served={info.get('vessels_served', 0)}/{len(env.vessels)}, "
                f"epsilon={agent.epsilon:.3f}"
            )
    print(f"Training complete. Final epsilon: {agent.epsilon:.3f}")
    return agent, history


def evaluate_rl_agent(env, agent, episodes: int = 100):
    """Evaluate a greedy policy and retain assignments from the first episode."""
    original_epsilon = agent.epsilon
    agent.epsilon = 0.0
    rewards, served_counts, completion_times = [], [], []
    assignment_details = []
    try:
        for episode in range(episodes):
            state = env.reset()
            total_reward = 0.0
            while True:
                action = agent.act(state, env.get_valid_actions())
                state, reward, done, info = env.step(action)
                total_reward += reward
                if done:
                    rewards.append(total_reward)
                    served_counts.append(info["vessels_served"])
                    completion_times.append(info["total_time"])
                    if episode == 0:
                        assignment_details = copy.deepcopy(env.assignment_history)
                    break
    finally:
        agent.epsilon = original_epsilon

    results = {
        "avg_reward": float(np.mean(rewards)),
        "std_reward": float(np.std(rewards)),
        "avg_vessels_served": float(np.mean(served_counts)),
        "avg_completion_time": float(np.mean(completion_times)),
        "assignment_details": assignment_details,
    }
    print("Greedy policy evaluation")
    print(f"Average reward: {results['avg_reward']:.2f} ± {results['std_reward']:.2f}")
    print(
        f"Average vessels served: {results['avg_vessels_served']:.2f}/"
        f"{len(env.vessels)}"
    )
    print(f"Average episode end time: {results['avg_completion_time']:.2f} h")
    return results

print("Training and evaluation functions defined.")

## 5. Train on the fixed instance

The episode count, horizon, and instance data are explicit inputs. Training time is measured for reporting only and does not affect the environment. Because no random seed is imposed, exploration and (for DQN) network initialization can produce different results across runs.

In [ ]:
env = DynamicBerthEnvironment(vessels, berths, max_time_horizon=24)
print(f"State vector length: {env.state_size}")
print(f"Number of actions: {env.action_space_n} (including wait)")
print(f"Time horizon: {env.max_time} h")

start_time = time.time()
trained_agent, training_history = train_rl_agent(env, episodes=500, max_steps=100)
training_time = (time.time() - start_time) / 60
print(f"Training time: {training_time:.2f} minutes")
eval_results = evaluate_rl_agent(env, trained_agent, episodes=100)
print("Training and evaluation completed.")

## 6. Inspect training metrics

The panels show episode reward, number of vessels served, exploration rate, and episode length. A moving average makes broad trends easier to see, but it does not replace the underlying episode data. In particular, a changing reward curve is not by itself evidence of convergence; service counts and evaluation results should be considered alongside it.

In [ ]:
def visualize_rl_training(training_history, eval_results):
    """Plot training metrics and print a concise descriptive summary."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("Reinforcement Learning Training Analysis", fontsize=15)
    episodes = np.arange(len(training_history["episode_rewards"]))
    window = 50

    metrics = [
        ("episode_rewards", "Episode reward", "Reward"),
        ("vessels_served", "Vessels served", "Vessels"),
        ("epsilon_history", "Exploration rate", "Epsilon"),
        ("episode_lengths", "Episode length", "Steps"),
    ]
    for ax, (key, title, ylabel) in zip(axes.flat, metrics):
        values = np.asarray(training_history[key])
        ax.plot(episodes, values, alpha=0.55, linewidth=1, label="Episode")
        if len(values) >= window:
            average = np.convolve(values, np.ones(window) / window, mode="valid")
            ax.plot(
                np.arange(window - 1, len(values)), average,
                linewidth=2, label=f"{window}-episode moving average"
            )
        ax.set(title=title, xlabel="Episode", ylabel=ylabel)
        ax.grid(True, alpha=0.3)
        ax.legend()
    axes[1, 0].set_ylim(0, 1.05)
    axes[0, 1].set_ylim(-0.1, len(vessels) + 0.1)
    plt.tight_layout()
    plt.show()

    rewards = training_history["episode_rewards"]
    served = training_history["vessels_served"]
    print("Training summary")
    print(f"Mean reward, first 100 episodes: {np.mean(rewards[:100]):.2f}")
    print(f"Mean reward, last 100 episodes: {np.mean(rewards[-100:]):.2f}")
    print(f"Mean served, first 100 episodes: {np.mean(served[:100]):.2f}")
    print(f"Mean served, last 100 episodes: {np.mean(served[-100:]):.2f}")
    print(f"Greedy evaluation mean reward: {eval_results['avg_reward']:.2f}")

visualize_rl_training(training_history, eval_results)

## 7. Examine the learned policy

A policy is the rule that selects an action from the current state. This inspection records the greedy action, waiting queue, valid actions, and Q-values at each decision point. The action-confidence measure is the difference between the largest and second-largest Q-values among valid actions; it measures separation in the agent's estimates, not statistical certainty that the chosen action is best.

In [ ]:
def analyze_learned_policy(env, agent):
    """Record and visualize greedy decisions for one episode."""
    original_epsilon = agent.epsilon
    agent.epsilon = 0.0
    decisions = []
    try:
        state = env.reset()
        while True:
            valid_actions = env.get_valid_actions()
            if DEEP_LEARNING_AVAILABLE:
                was_training = agent.q_network.training
                agent.q_network.eval()
                with torch.no_grad():
                    q_values = agent.q_network(
                        torch.as_tensor(state, dtype=torch.float32)
                    ).cpu().numpy()
                agent.q_network.train(was_training)
            else:
                q_values = agent.q_table[agent._state_to_key(state)].copy()

            action = agent.act(state, valid_actions)
            valid_values = np.asarray([q_values[a] for a in valid_actions])
            ordered = np.sort(valid_values)
            confidence = ordered[-1] - ordered[-2] if len(ordered) > 1 else 0.0
            decision = {
                "time": env.current_time,
                "waiting_queue": [v.id for v in env.waiting_queue],
                "valid_actions": valid_actions,
                "chosen_action": action,
                "q_values": q_values.copy(),
                "best_q": float(valid_values.max()),
                "action_confidence": float(confidence),
            }
            if action == env.wait_action:
                decision["action_type"] = "wait"
            else:
                vessel_idx, berth_idx = divmod(action, env.n_berths)
                vessel = env.vessels[vessel_idx]
                decision.update(
                    action_type="assign",
                    vessel_id=vessel.id,
                    berth_id=env.berths[berth_idx].id,
                    vessel_teu=vessel.teu_capacity,
                    vessel_deadline=vessel.desired_departure,
                )
            decisions.append(decision)
            state, _, done, _ = env.step(action)
            if done:
                break
    finally:
        agent.epsilon = original_epsilon

    df_decisions = pd.DataFrame(decisions)
    counts = df_decisions["action_type"].value_counts()
    print("Greedy policy decision summary")
    print(f"Total decisions: {len(df_decisions)}")
    print(f"Assignments: {counts.get('assign', 0)}")
    print(f"Waits: {counts.get('wait', 0)}")

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("Learned Policy Analysis")
    action_values = df_decisions["action_type"].map({"assign": 1, "wait": 0})
    colors = action_values.map({1: "tab:green", 0: "tab:blue"})
    axes[0, 0].scatter(df_decisions["time"], action_values, c=colors, alpha=0.75)
    axes[0, 0].set(
        xlabel="Time (hours)", ylabel="Decision type", title="Decision timeline",
        yticks=[0, 1], yticklabels=["Wait", "Assign"]
    )
    axes[0, 1].plot(df_decisions["time"], df_decisions["best_q"], color="purple")
    axes[0, 1].set(xlabel="Time (hours)", ylabel="Q-value", title="Best valid-action Q-value")
    axes[1, 0].plot(
        df_decisions["time"], df_decisions["action_confidence"], color="darkorange"
    )
    axes[1, 0].set(
        xlabel="Time (hours)", ylabel="Q-value difference",
        title="Separation between top valid actions"
    )
    queue_sizes = [len(queue) for queue in df_decisions["waiting_queue"]]
    axes[1, 1].plot(df_decisions["time"], queue_sizes, color="tab:red")
    axes[1, 1].set(
        xlabel="Time (hours)", ylabel="Vessels", title="Waiting queue size"
    )
    for ax in axes.flat:
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return df_decisions

policy_analysis = analyze_learned_policy(env, trained_agent)

## 8. Compare with reference policies

The comparison uses the same fixed environment and reward definition. The random policy samples uniformly from valid actions. The EDF (earliest-deadline-first) reference chooses the waiting vessel with the earliest desired departure, then selects a feasible available berth with the earliest completion time. These references provide context for the learned policy; they do not change the Q-learning method or provide an optimality certificate.

All policies are evaluated over repeated episodes. Since this example has fixed data and deterministic transitions, differences between episodes arise from exploration in the random policy or from the learned agent's action selection, not from random vessel arrivals.

In [ ]:
def compare_rl_vs_baselines(env, agent, test_episodes: int = 50):
    """Compare the learned policy with EDF and random valid-action policies."""
    results = {}
    original_epsilon = agent.epsilon
    agent.epsilon = 0.0

    def run_episode(policy):
        state = env.reset()
        total_reward = 0.0
        while True:
            action = policy(state)
            state, reward, done, info = env.step(action)
            total_reward += reward
            if done:
                return total_reward, info["vessels_served"], info["total_time"]

    try:
        for name in ("RL Agent", "EDF Heuristic", "Random Policy"):
            episode_results = []
            for _ in range(test_episodes):
                if name == "RL Agent":
                    policy = lambda state: agent.act(state, env.get_valid_actions())
                elif name == "Random Policy":
                    policy = lambda state: random.choice(env.get_valid_actions())
                else:
                    def policy(state):
                        if not env.waiting_queue:
                            return env.wait_action
                        vessel = min(
                            env.waiting_queue,
                            key=lambda item: (item.desired_departure, item.id),
                        )
                        vessel_idx = env.vessels.index(vessel)
                        choices = []
                        for berth_idx, berth in enumerate(env.berths):
                            if (
                                vessel.length <= berth.capacity
                                and env.berth_availability[berth.id] <= env.current_time
                            ):
                                completion = (
                                    env.current_time + vessel.processing_times[berth.id]
                                )
                                choices.append((completion, berth_idx))
                        if not choices:
                            return env.wait_action
                        _, berth_idx = min(choices)
                        return vessel_idx * env.n_berths + berth_idx
                episode_results.append(run_episode(policy))
            results[name] = np.asarray(episode_results, dtype=float)
    finally:
        agent.epsilon = original_epsilon

    rows = []
    for name, values in results.items():
        rows.append(
            [
                name,
                f"{values[:, 0].mean():.2f} ± {values[:, 0].std():.2f}",
                f"{values[:, 1].mean():.2f}",
                f"{values[:, 2].mean():.2f}",
            ]
        )
    comparison = pd.DataFrame(
        rows,
        columns=["Method", "Mean reward ± standard deviation", "Mean vessels served", "Mean episode end time (h)"],
    )
    print(comparison.to_string(index=False))

    names = list(results)
    rewards = [results[name][:, 0] for name in names]
    served = [results[name][:, 1].mean() for name in names]
    times = [results[name][:, 2].mean() for name in names]
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("Learned Policy and Reference Policy Comparison")
    axes[0, 0].bar(
        names, [values.mean() for values in rewards],
        yerr=[values.std() for values in rewards], capsize=4,
        color=["tab:blue", "tab:green", "tab:red"]
    )
    axes[0, 0].set(title="Mean reward", ylabel="Reward")
    axes[0, 1].bar(names, served, color=["tab:blue", "tab:green", "tab:red"])
    axes[0, 1].set(title="Vessels served", ylabel="Mean vessels")
    axes[0, 1].set_ylim(0, len(env.vessels) + 0.5)
    axes[1, 0].bar(names, times, color=["tab:blue", "tab:green", "tab:red"])
    axes[1, 0].set(title="Episode end time", ylabel="Hours")
    box = axes[1, 1].boxplot(rewards, tick_labels=names, patch_artist=True)
    for patch, color in zip(box["boxes"], ["tab:blue", "tab:green", "tab:red"]):
        patch.set_facecolor(color)
        patch.set_alpha(0.65)
    axes[1, 1].set(title="Reward distributions", ylabel="Reward")
    for ax in axes.flat:
        ax.tick_params(axis="x", rotation=15)
        ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
    return comparison, results

comparison_df, comparison_results = compare_rl_vs_baselines(env, trained_agent)

## 9. Interpretation, strengths, and limitations

### What the experiment can show
- **Sequential decisions:** The agent selects assignments and wait actions using the current encoded state.
- **Reward-guided learning:** Q-values estimate discounted future reward, so an action is judged by both its immediate reward and the value of its possible next states.
- **Exploration and exploitation:** Epsilon-greedy selection explores valid actions early in training and increasingly favors estimated high-value actions as epsilon decreases.
- **Empirical policy assessment:** Episode metrics and reference-policy comparisons describe performance on the specified environment.

### What it cannot establish
- **No optimality guarantee:** A finite run, approximate state representation, and learned value estimates do not prove that the policy is optimal.
- **Limited generalization evidence:** Evaluation on the same fixed instance does not test unseen vessel sets, stochastic arrivals, or changing operating conditions.
- **Reward dependence:** The policy responds to the chosen reward weights; changing them changes the trade-offs it is asked to learn.
- **Interpretability limits:** Q-values and neural-network parameters do not by themselves provide a causal explanation for a decision.
- **Training cost and sensitivity:** Learning behavior can depend on exploration, learning rate, discount factor, state representation, and other hyperparameters.

### Modeling notes and possible extensions
The example treats berth availability as a single time value per berth and does not model spatial separation, simultaneous constraints between adjacent vessels, or uncertainty in service durations. Vessel dimensions are represented by length alone for feasibility; TEU is descriptive. The wait action advances time in one-hour increments, while an assignment occupies its berth for its full processing duration. Natural extensions include richer operational constraints, stochastic arrivals or processing, and evaluation on independently generated instances. Those extensions require a corresponding state, transition, and reward design; they are not implemented here.